# Raw Layer Transformations

In [0]:
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("skipRows", 2) \
.load("/Volumes/workspace/gold_sales_bronze/volumes/gold_sales_transactions.csv")

In [0]:
display(df)

transaction_id,transaction_timestamp,branch_id,supplier_id,product_type,weight_troy_oz,spot_price_usd_oz,premium_pct,sale_price_usd,currency_code,exchange_rate,sales_channel,transaction_status,customer_segment,audit_metadata_json
TXN-000001,2024-06-30T19:25:00.000Z,BR-004,SUP-004,18K Coin,20.0438,"$1,985.19",15.05,"$45,779.26",SGD,1.34,In-Store,Returned,Retail,"""{""""transaction_id"""":""""TXN-000001"""""
TXN-000002,2024-07-09T23:22:00.000Z,BR-019,SUP-003,22K Bar,41.4292,"$1,991.96",5.18,"$86,800.12",USD,1.0,Auction,Completed,Wholesale,"""{""""transaction_id"""":""""TXN-000002"""""
TXN-000003,2024-07-23T07:10:00.000Z,BR-019,SUP-008,24K Coin,28.2327,"$2,018.35",3.2,"$58,806.94",AED,3.673,Auction,Pending,Institutional,"""{""""transaction_id"""":""""TXN-000003"""""
TXN-000004,2024-08-11T01:52:00.000Z,BR-008,SUP-005,18K Jewellery,23.5473,"$2,035.59",7.47,"$51,513.22",AED,3.673,Auction,Completed,HNI,"""{""""transaction_id"""":""""TXN-000004"""""
TXN-000005,2024-06-12T01:55:00.000Z,BR-016,SUP-010,Bullion Round,21.1971,"$2,002.61",4.01,"$44,151.75",GBP,0.79,Wholesale,Returned,Retail,"""{""""transaction_id"""":""""TXN-000005"""""
TXN-000006,2024-08-12T20:50:00.000Z,BR-003,SUP-001,Gold ETF Unit,3.9038,"$2,006.17",5.95,"$8,297.67",USD,1.0,Wholesale,Completed,Institutional,"""{""""transaction_id"""":""""TXN-000006"""""
TXN-000007,2024-06-16T15:06:00.000Z,BR-005,SUP-015,Sovereign Bond,6.2962,"$1,979.27",14.18,"$14,228.97",AED,3.673,Online,Pending,Wholesale,"""{""""transaction_id"""":""""TXN-000007"""""
TXN-000008,2024-08-18T23:24:00.000Z,BR-013,SUP-014,Bullion Round,14.2818,"$2,020.45",7.42,"$30,996.75",AED,3.673,Auction,Completed,HNI,"""{""""transaction_id"""":""""TXN-000008"""""
TXN-000009,2024-06-25T05:58:00.000Z,BR-019,SUP-007,24K Coin,0.7026,"$1,996.28",7.59,"$1,509.04",SGD,1.34,In-Store,Pending,HNI,"""{""""transaction_id"""":""""TXN-000009"""""
TXN-000010,2024-07-29T01:37:00.000Z,BR-006,SUP-006,22K Jewellery,3.5813,"$1,998.92",4.94,"$7,512.37",USD,1.0,Auction,Completed,Institutional,"""{""""transaction_id"""":""""TXN-000010"""""


In [0]:
from pyspark.sql.functions import col, regexp_replace, current_timestamp

df_final = df.withColumn("spot_price_usd_oz", regexp_replace(col("spot_price_usd_oz"), "[$,]", "")) \
    .withColumn("sale_price_usd", regexp_replace(col("sale_price_usd"), "[$,]", "")) \
    .withColumn("transaction_id", col("transaction_id").cast("string")) \
    .withColumn("transaction_timestamp", col("transaction_timestamp").cast("timestamp")) \
    .withColumn("branch_id", col("branch_id").cast("string")) \
    .withColumn("supplier_id", col("supplier_id").cast("string")) \
    .withColumn("product_type", col("product_type").cast("string")) \
    .withColumn("weight_troy_oz", col("weight_troy_oz").cast("double")) \
    .withColumn("spot_price_usd_oz", col("spot_price_usd_oz").cast("double")) \
    .withColumn("premium_pct", col("premium_pct").cast("double")) \
    .withColumn("sale_price_usd", col("sale_price_usd").cast("double")) \
    .withColumn("currency_code", col("currency_code").cast("string")) \
    .withColumn("exchange_rate", col("exchange_rate").cast("double")) \
    .withColumn("sales_channel", col("sales_channel").cast("string")) \
    .withColumn("transaction_status", col("transaction_status").cast("string")) \
    .withColumn("customer_segment", col("customer_segment").cast("string")) \
.withColumn("ingestion_timestamp", current_timestamp())

In [0]:
df_final.columns

['transaction_id',
 'transaction_timestamp',
 'branch_id',
 'supplier_id',
 'product_type',
 'weight_troy_oz',
 'spot_price_usd_oz',
 'premium_pct',
 'sale_price_usd',
 'currency_code',
 'exchange_rate',
 'sales_channel',
 'transaction_status',
 'customer_segment',
 'audit_metadata_json',
 'ingestion_timestamp']

In [0]:
df_final.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold_sales_bronze.raw_transactions")

In [0]:
%sql
select * from gold_sales_bronze.raw_transactions;

transaction_id,transaction_timestamp,branch_id,supplier_id,product_type,weight_troy_oz,spot_price_usd_oz,premium_pct,sale_price_usd,currency_code,exchange_rate,sales_channel,transaction_status,customer_segment,audit_metadata_json,ingestion_timestamp
TXN-000001,2024-06-30T19:25:00.000Z,BR-004,SUP-004,18K Coin,20.0438,1985.19,15.05,45779.26,SGD,1.34,In-Store,Returned,Retail,"""{""""transaction_id"""":""""TXN-000001""""",2026-05-01T10:11:00.750Z
TXN-000002,2024-07-09T23:22:00.000Z,BR-019,SUP-003,22K Bar,41.4292,1991.96,5.18,86800.12,USD,1.0,Auction,Completed,Wholesale,"""{""""transaction_id"""":""""TXN-000002""""",2026-05-01T10:11:00.750Z
TXN-000003,2024-07-23T07:10:00.000Z,BR-019,SUP-008,24K Coin,28.2327,2018.35,3.2,58806.94,AED,3.673,Auction,Pending,Institutional,"""{""""transaction_id"""":""""TXN-000003""""",2026-05-01T10:11:00.750Z
TXN-000004,2024-08-11T01:52:00.000Z,BR-008,SUP-005,18K Jewellery,23.5473,2035.59,7.47,51513.22,AED,3.673,Auction,Completed,HNI,"""{""""transaction_id"""":""""TXN-000004""""",2026-05-01T10:11:00.750Z
TXN-000005,2024-06-12T01:55:00.000Z,BR-016,SUP-010,Bullion Round,21.1971,2002.61,4.01,44151.75,GBP,0.79,Wholesale,Returned,Retail,"""{""""transaction_id"""":""""TXN-000005""""",2026-05-01T10:11:00.750Z
TXN-000006,2024-08-12T20:50:00.000Z,BR-003,SUP-001,Gold ETF Unit,3.9038,2006.17,5.95,8297.67,USD,1.0,Wholesale,Completed,Institutional,"""{""""transaction_id"""":""""TXN-000006""""",2026-05-01T10:11:00.750Z
TXN-000007,2024-06-16T15:06:00.000Z,BR-005,SUP-015,Sovereign Bond,6.2962,1979.27,14.18,14228.97,AED,3.673,Online,Pending,Wholesale,"""{""""transaction_id"""":""""TXN-000007""""",2026-05-01T10:11:00.750Z
TXN-000008,2024-08-18T23:24:00.000Z,BR-013,SUP-014,Bullion Round,14.2818,2020.45,7.42,30996.75,AED,3.673,Auction,Completed,HNI,"""{""""transaction_id"""":""""TXN-000008""""",2026-05-01T10:11:00.750Z
TXN-000009,2024-06-25T05:58:00.000Z,BR-019,SUP-007,24K Coin,0.7026,1996.28,7.59,1509.04,SGD,1.34,In-Store,Pending,HNI,"""{""""transaction_id"""":""""TXN-000009""""",2026-05-01T10:11:00.750Z
TXN-000010,2024-07-29T01:37:00.000Z,BR-006,SUP-006,22K Jewellery,3.5813,1998.92,4.94,7512.37,USD,1.0,Auction,Completed,Institutional,"""{""""transaction_id"""":""""TXN-000010""""",2026-05-01T10:11:00.750Z


# Refind Layer Transformations

In [0]:
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
.load("/Volumes/workspace/gold_sales_bronze/volumes/spot_price_feed.csv")

In [0]:
display(df)

price_date,spot_usd_oz,spot_aed_oz,spot_sgd_oz,spot_gbp_oz,spot_inr_oz
2024-06-01,"$1,917.86","$7,044.30","$2,569.93","$1,515.11","$1,60,141.31"
2024-06-02,"$1,918.64","$7,047.16","$2,570.98","$1,515.73","$1,60,206.44"
2024-06-03,"$1,921.63","$7,058.15","$2,574.98","$1,518.09","$1,60,456.11"
2024-06-04,"$1,913.26","$7,027.40","$2,563.77","$1,511.48","$1,59,757.21"
2024-06-05,"$1,930.53","$7,090.84","$2,586.91","$1,525.12","$1,61,199.26"
2024-06-06,"$1,913.85","$7,029.57","$2,564.56","$1,511.94","$1,59,806.48"
2024-06-07,"$1,926.59","$7,076.37","$2,581.63","$1,522.01","$1,60,870.26"
2024-06-08,"$1,914.56","$7,032.18","$2,565.51","$1,512.50","$1,59,865.76"
2024-06-09,"$1,930.32","$7,090.07","$2,586.63","$1,524.95","$1,61,181.72"
2024-06-10,"$1,928.75","$7,084.30","$2,584.53","$1,523.71","$1,61,050.62"


In [0]:
from pyspark.sql.functions import col, regexp_replace, current_timestamp

df_final = df \
    .withColumn("spot_usd_oz", regexp_replace(col("spot_usd_oz"), "[$,]", "").cast("double")) \
    .withColumn("spot_aed_oz", regexp_replace(col("spot_aed_oz"), "[$,]", "").cast("double")) \
    .withColumn("spot_sgd_oz", regexp_replace(col("spot_sgd_oz"), "[$,]", "").cast("double")) \
    .withColumn("spot_gbp_oz", regexp_replace(col("spot_gbp_oz"), "[$,]", "").cast("double")) \
    .withColumn("spot_inr_oz", regexp_replace(col("spot_inr_oz"), "[$,]", "").cast("double")) \
    .withColumn("ingestion_timestamp", current_timestamp())

In [0]:
df_final.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold_sales_bronze.raw_spot_prices")

In [0]:
display(df_final)

price_date,spot_usd_oz,spot_aed_oz,spot_sgd_oz,spot_gbp_oz,spot_inr_oz,ingestion_timestamp
2024-06-01,1917.86,7044.3,2569.93,1515.11,160141.31,2026-05-01T10:36:42.812Z
2024-06-02,1918.64,7047.16,2570.98,1515.73,160206.44,2026-05-01T10:36:42.812Z
2024-06-03,1921.63,7058.15,2574.98,1518.09,160456.11,2026-05-01T10:36:42.812Z
2024-06-04,1913.26,7027.4,2563.77,1511.48,159757.21,2026-05-01T10:36:42.812Z
2024-06-05,1930.53,7090.84,2586.91,1525.12,161199.26,2026-05-01T10:36:42.812Z
2024-06-06,1913.85,7029.57,2564.56,1511.94,159806.48,2026-05-01T10:36:42.812Z
2024-06-07,1926.59,7076.37,2581.63,1522.01,160870.26,2026-05-01T10:36:42.812Z
2024-06-08,1914.56,7032.18,2565.51,1512.5,159865.76,2026-05-01T10:36:42.812Z
2024-06-09,1930.32,7090.07,2586.63,1524.95,161181.72,2026-05-01T10:36:42.812Z
2024-06-10,1928.75,7084.3,2584.53,1523.71,161050.62,2026-05-01T10:36:42.812Z


# Refined_Layer_Enriched_transactions

In [0]:
from pyspark.sql.functions import col, from_json, when
from pyspark.sql.types import StructType, StringType, DoubleType

df_txn = spark.table("workspace.gold_sales_bronze.raw_transactions")
df_branch = spark.table("workspace.gold_sales_bronze.branch_master")

audit_schema = StructType() \
    .add("lab_name", StringType()) \
    .add("purity", DoubleType()) \
    .add("inspector", StringType())


df_silver = df_txn \
    .join(df_branch, on="branch_id", how="left") \
    .withColumn("audit_parsed", from_json(col("audit_metadata_json"), audit_schema)) \
    .select("*", "audit_parsed.*") \
    .drop("audit_parsed") \
    .withColumn(
        "gross_revenue_usd",
        col("weight_troy_oz") * col("spot_price_usd_oz") * (1 + col("premium_pct") / 100)
    ) \
    .withColumn(
        "gold_category",
        when(col("product_type").like("%bar%"), "Bar")
        .when(col("product_type").like("%coin%"), "Coin")
        .when(col("product_type").like("%jewellery%"), "Jewellery")
        .otherwise("Other")
    )

In [0]:
df_silver.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold_sales_silver.enriched_transactions")

branch_master

In [0]:
from pyspark.sql.functions import col, lower, initcap, to_date, when

df_branch_clean = df_branch \
    .withColumn("region", initcap(lower(col("region")))) \
    .withColumn("country", initcap(lower(col("country")))) \
    .withColumn("opened_date", to_date(col("opened_date"))) \
    .withColumn(
        "is_active",
        when(lower(col("is_active")).isin("y", "yes", "1", "true"), True)
        .otherwise(False)
    )

In [0]:
df_branch_clean.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold_sales_silver.branch_master")

# Consolidated_Layer

In [0]:
from pyspark.sql.functions import col, to_date, sum, avg, count

df_txn = spark.table("workspace.gold_sales_silver.enriched_transactions").alias("t")
df_branch = spark.table("workspace.gold_sales_silver.branch_master").alias("b")

df_gold = df_txn \
    .join(df_branch, col("t.branch_id") == col("b.branch_id"), "left") \
    .withColumn("sale_date", to_date(col("t.transaction_timestamp"))) \
    .groupBy(
        col("sale_date"),
        col("t.branch_id"),
        col("b.region"),         
        col("t.gold_category")
    ) \
    .agg(
        sum(col("t.weight_troy_oz")).alias("total_weight_oz"),
        sum(col("t.gross_revenue_usd")).alias("total_revenue_usd"),
        avg(col("t.premium_pct")).alias("avg_premium_pct"),
        count("*").alias("transaction_count")
    )

In [0]:
df_gold.write.format("delta") \
    .mode("overwrite") \
    .partitionBy("sale_date") \
    .saveAsTable("workspace.gold_sales_gold.daily_branch_revenue")